# 호우날두 — 6시간 최대 하천수위 상승량 예측 (Baseline v1)

**누수 방지 규칙 (Q&A 9/23, 10/6 기준)**
- 행 t 예측에는 **시점 t 이하 정보만** 사용 → 같은 관측소의 t+1~t+6 행 `wl_t` 참조 금지
- 허용: 같은 관측소의 이전 시각 행 / 같은 호우사상 다른 관측소의 시점 t 및 t 이전 값
- 금지: 사상 종료시점·전체 사상 길이 등 **미래 의존 정보** 생성
- 관측소 간 관계·통계는 **train으로** 구성
- 검증: 호우사상 단위 GroupKFold 4-fold (주최측 기준선과 동일 조건)

| 주최측 기준선 (Train 4-fold OOF) | RMSE |
|---|---|
| 변화 없음 | 0.410 |
| 과거 수위만 | 0.275 |
| 강우·기상 + 유역환경 | 0.244 |

> **USE_COMMON_FOLDS = True (기본)**: `outputs/common/event_folds.csv`의 fold를 쓰고 홀드아웃 18개 사상을 뺀 dev 104개 사상으로 CV → 수아·한동 노트북과 점수 비교 가능 (수아 1차 실험 '기준' 0.2182와 같아야 함)
> `False`: 122개 사상 전체로 GroupKFold를 직접 만듦. **전체 참고용**이며 다른 노트북과 점수를 직접 비교하지 않음

## STEP 0. 환경 설정

In [ ]:
%pip install lightgbm

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

# 레포 루트 또는 notebooks/ 어디서 실행해도 data/를 찾음
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_DIR = ROOT / "data"
OUT_DIR = DATA_DIR.parent / "outputs"
OUT_DIR.mkdir(exist_ok=True)

SEED = 42
N_FOLDS = 4
TARGET = "target_maxrise_6h"
USE_COMMON_FOLDS = True   # True: 공통 fold·홀드아웃 사용 (팀 비교용) / False: 전체 122개 사상 참고용
EXPECTED = {"train_rows": 749_657, "test_rows": 186_263, "train_events": 122, "test_events": 30}

## STEP 1. 데이터 로드

In [ ]:
def find_file(stem):
    hits = sorted(p for p in DATA_DIR.glob(stem + "*") if not p.name.startswith("~$"))
    if not hits:
        raise FileNotFoundError(f"{stem} 파일을 {DATA_DIR}에서 찾지 못함")
    return hits[0]

def read_table(path):
    if path.suffix in (".xlsx", ".xls"):
        return pd.read_excel(path)
    for enc in ("utf-8-sig", "cp949"):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"인코딩 확인 필요: {path}")

train = pd.read_csv(DATA_DIR / "train.csv.gz")
test = pd.read_csv(DATA_DIR / "test.csv.gz")
sub = read_table(find_file("sample_submission"))
ddict = read_table(find_file("data_dictionary"))

for df in (train, test):
    df["timestamp"] = pd.to_datetime(df["timestamp"])

print("train:", train.shape, "| test:", test.shape, "| sub:", sub.shape)
print("행 수 일치:", len(train) == EXPECTED["train_rows"], len(test) == EXPECTED["test_rows"])
print("test 관측소 중 train에 없는 관측소:", len(set(test.station_id) - set(train.station_id)))
ddict

## STEP 2. 호우사상 ID 복원
- 사상 번호가 없으므로 **timestamp가 1시간 넘게 끊기는 지점**으로 사상을 나눔 (Q&A 10/6 허용 방식)
- 시점 t의 사상 판정은 "t-1 시각이 존재하는가"만 보므로 과거 정보만 사용
- 사상 길이·종료시점은 만들지 않음

In [ ]:
def add_event_id(df, prefix, gap_h=1):
    ts = pd.Series(np.sort(df["timestamp"].unique()))
    event_no = ts.diff().gt(pd.Timedelta(hours=gap_h)).cumsum()
    m = pd.DataFrame({"timestamp": ts, "event_id": prefix + event_no.astype(str).str.zfill(3)})
    return df.merge(m, on="timestamp", how="left")

train = add_event_id(train, "TR_E")
test = add_event_id(test, "TE_E")

print("train 사상 수:", train.event_id.nunique(), "(기대", EXPECTED["train_events"], ")")
print("test  사상 수:", test.event_id.nunique(), "(기대", EXPECTED["test_events"], ")")
# 개수가 다르면 사상 사이 간격이 1시간인 경우가 있다는 뜻 → gap_h 조정 또는 wl 연속성으로 추가 검증 필요

### 2-1. 공통 fold·홀드아웃 불러오기
`jaehyuk_pipeline.ipynb`가 만든 `outputs/common/event_folds.csv` 사용. 노트북 안에서 fold를 새로 만들지 않음.

In [ ]:
if USE_COMMON_FOLDS:
    ef = pd.read_csv(OUT_DIR / "common" / "event_folds.csv")
    # 사상 ID 복원이 공통 파일과 같은지 확인 (사상별 행 수 비교)
    my_counts = train.groupby("event_id").size()
    assert my_counts.reindex(ef["event_id"]).tolist() == ef["n_rows"].tolist(), "사상 ID 복원 결과가 공통 파일과 다름"
    train = train.merge(ef[["event_id", "fold"]], on="event_id", how="left")
    hold_events = set(ef.loc[ef["fold"] == -1, "event_id"])
    print(f"홀드아웃 {len(hold_events)}개 사상 / dev {ef['fold'].ge(0).sum()}개 사상")
else:
    hold_events = set()
    print("전체 참고용 모드: 홀드아웃 없이 122개 사상 전체 사용")

## STEP 3. 기준선 확인 (전량 0 예측)

In [ ]:
def rmse(y, p):
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(p)) ** 2)))

print("변화 없음 RMSE:", round(rmse(train[TARGET], 0), 4), "(주최측 0.410)")

ev = train.groupby("event_id")[TARGET].apply(lambda s: rmse(s, 0))
print("사상별 RMSE (변화 없음) 분포")
ev.describe().round(3)

## STEP 4. 피처 생성 (시점 t 이하 정보만)
| 그룹 | 내용 | 근거 |
|---|---|---|
| A. 수위 변화 | `wl_t - wl_t_minus_Xh`, 가속도, 직전 6h 실현 상승량 | 같은 행 + 같은 관측소 이전 행 |
| B. 강우 이력 | 1h 강우 lag, 강우 추세, 사상 내 누적·최대 강우 | 같은 관측소 이전 행 |
| C. 사상 맥락 | 사상 시작 이후 경과시간, 사상 시작 대비 수위 변화 | 사상 시작은 과거 시점 |
| D. 동시각 공간 | 같은 사상·같은 시각 전체 관측소 평균 강우·수위변화 | Q&A 10/6 (나) 허용 |
| E. 관측소 정규화 | train 기준 관측소별 수위 중앙값·IQR로 상대수위 | 관측소 통계는 train으로 |

In [ ]:
EPS = 1e-6

def make_features(df, station_stats):
    df = df.sort_values(["station_id", "timestamp"]).reset_index(drop=True)
    g = df.groupby(["station_id", "event_id"], sort=False)

    def lag(col, k):
        # 정확히 k시간 전 행일 때만 사용 (결측 시각이 있으면 NaN)
        ok = (df["timestamp"] - g["timestamp"].shift(k)) == pd.Timedelta(hours=k)
        return g[col].shift(k).where(ok)

    # A. 수위 변화
    for h in (1, 3, 6, 12, 24):
        df[f"dwl_{h}h"] = df["wl_t"] - df[f"wl_t_minus_{h}h"]
    df["accel_wl"] = df["dwl_1h"] - (df["wl_t_minus_1h"] - df["wl_t_minus_3h"]) / 2
    wl_l2, wl_l4, wl_l5 = lag("wl_t", 2), lag("wl_t", 4), lag("wl_t", 5)
    past_max = pd.concat([df["wl_t"], df["wl_t_minus_1h"], wl_l2, df["wl_t_minus_3h"], wl_l4, wl_l5], axis=1).max(axis=1)
    df["past_maxrise_6h"] = past_max - df["wl_t_minus_6h"]      # 직전 6시간 실현 상승량 (t 이하 값만)
    df["wl_below_past_max"] = df["wl_t"] - past_max             # 최근 고점 대비 위치

    # B. 강우 이력
    for c in ("imerg_rain_1h", "sfc_rain_1h", "radar_reflectivity_mean", "radar_reflectivity_ge20_fraction"):
        for k in (1, 2, 3):
            df[f"{c}_lag{k}"] = lag(c, k)
    df["imerg_trend_1h"] = df["imerg_rain_1h"] - df["imerg_rain_1h_lag1"]
    df["sfc_trend_1h"] = df["sfc_rain_1h"] - df["sfc_rain_1h_lag1"]
    for src in ("imerg", "sfc"):
        df[f"{src}_front_1_6"] = df[f"{src}_rain_1h"] / (df[f"{src}_rain_6h"] + EPS)
        df[f"{src}_front_6_24"] = df[f"{src}_rain_6h"] / (df[f"{src}_rain_24h"] + EPS)
        df[f"{src}_rain_7_24h"] = df[f"{src}_rain_24h"] - df[f"{src}_rain_6h"]
        df[f"{src}_event_cum"] = g[f"{src}_rain_1h"].cumsum()
        df[f"{src}_event_max1h"] = g[f"{src}_rain_1h"].cummax()
    df["imerg_minus_sfc_6h"] = df["imerg_rain_6h"] - df["sfc_rain_6h"]

    # C. 사상 맥락
    df["event_elapsed_h"] = (df["timestamp"] - df.groupby("event_id")["timestamp"].transform("min")).dt.total_seconds() / 3600
    df["wl_since_event_start"] = df["wl_t"] - g["wl_t"].transform("first")

    # D. 동시각 공간 (같은 사상·같은 시각)
    gt = df.groupby(["event_id", "timestamp"])
    for c in ("imerg_rain_1h", "imerg_rain_6h", "sfc_rain_6h", "dwl_1h", "dwl_3h", "dwl_6h"):
        df[f"xs_mean_{c}"] = gt[c].transform("mean")
    df["xs_rising_frac"] = gt["dwl_1h"].transform(lambda s: (s > 0).mean())
    df["dwl_3h_vs_xs"] = df["dwl_3h"] - df["xs_mean_dwl_3h"]

    # E. 관측소 정규화 (train 통계)
    df = df.merge(station_stats, on="station_id", how="left")
    df["wl_rel"] = (df["wl_t"] - df["st_wl_med"]) / (df["st_wl_iqr"] + EPS)
    for h in (1, 3, 6):
        df[f"dwl_{h}h_scaled"] = df[f"dwl_{h}h"] / (df["st_wl_iqr"] + EPS)
    return df

# 관측소 통계는 dev 사상으로만 계산 (홀드아웃 정보가 섞이지 않게)
q = train[~train["event_id"].isin(hold_events)].groupby("station_id")["wl_t"].quantile([0.25, 0.5, 0.75]).unstack()
station_stats = pd.DataFrame({
    "station_id": q.index,
    "st_wl_med": q[0.5].values,
    "st_wl_iqr": (q[0.75] - q[0.25]).values,
})

tr = make_features(train, station_stats)
te = make_features(test, station_stats)
print("train feat:", tr.shape, "| test feat:", te.shape)

### 4-1. 누수 자가검증
각 사상의 **마지막 6시간 행을 지운 뒤** 피처를 다시 만들어, 남은 행의 피처값이 원래와 똑같은지 확인.
값이 하나라도 달라지면 그 피처는 미래 행을 참조하고 있다는 뜻.

In [ ]:
end = train.groupby("event_id")["timestamp"].transform("max")
cut = train[train["timestamp"] <= end - pd.Timedelta(hours=6)]
tr_cut = make_features(cut, station_stats).set_index("row_id")
tr_full = tr.set_index("row_id").loc[tr_cut.index]

feat_cols = [c for c in tr_cut.columns if c not in ("timestamp", "event_id", "fold", TARGET)]
diff = {c: int((~np.isclose(tr_full[c].astype(float), tr_cut[c].astype(float), equal_nan=True)).sum())
        for c in feat_cols if c != "station_id"}
leaky = {c: n for c, n in diff.items() if n > 0}
print("미래 의존 피처:", leaky if leaky else "없음 ✅")

## STEP 5. 사상 단위 4-fold CV (LightGBM)

In [ ]:
# 홀드아웃 사상은 CV에서 제외 (최종 조합 확정 후 한 번만 확인)
tr_all = tr
tr = tr[~tr["event_id"].isin(hold_events)].reset_index(drop=True)

if USE_COMMON_FOLDS:
    FOLDS = [(np.where(tr["fold"] != k)[0], np.where(tr["fold"] == k)[0]) for k in range(N_FOLDS)]
else:
    FOLDS = list(GroupKFold(n_splits=N_FOLDS).split(tr, tr[TARGET], tr["event_id"]))

DROP = {"row_id", "timestamp", "event_id", "fold", TARGET}
FEATURES = [c for c in tr.columns if c not in DROP]

USE_STATION_CAT = True   # station_id를 범주형으로 사용할지 (과적합 시 False로 비교)
if USE_STATION_CAT:
    cats = pd.CategoricalDtype(sorted(train.station_id.unique()))
    tr["station_id"] = tr["station_id"].astype(cats)
    te["station_id"] = te["station_id"].astype(cats)
else:
    FEATURES.remove("station_id")

PARAMS = dict(
    objective="regression", learning_rate=0.05, num_leaves=63,
    min_data_in_leaf=200, feature_fraction=0.8, bagging_fraction=0.8,
    bagging_freq=1, lambda_l2=1.0, seed=SEED, verbose=-1,
)

def run_cv(features, params=PARAMS, verbose=True):
    oof = np.zeros(len(tr))
    models = []
    for f, (i_tr, i_va) in enumerate(FOLDS):
        d_tr = lgb.Dataset(tr.iloc[i_tr][features], tr.iloc[i_tr][TARGET])
        d_va = lgb.Dataset(tr.iloc[i_va][features], tr.iloc[i_va][TARGET])
        m = lgb.train(params, d_tr, num_boost_round=5000, valid_sets=[d_va],
                      callbacks=[lgb.early_stopping(200, verbose=False)])
        oof[i_va] = m.predict(tr.iloc[i_va][features], num_iteration=m.best_iteration)
        models.append(m)
        if verbose:
            print(f"fold {f}: RMSE {rmse(tr.iloc[i_va][TARGET], oof[i_va]):.4f} | iter {m.best_iteration}")
    score = rmse(tr[TARGET], oof)
    if verbose:
        print(f"OOF Global RMSE: {score:.4f}  ({'dev 104개 사상, 수아 기준 0.2182와 비교' if USE_COMMON_FOLDS else '전체 참고용'} / 주최측 0.244)")
    return score, oof, models

cv_score, oof, models = run_cv(FEATURES)

### 5-1. 오차 진단 (사상별 RMSE · 피처 중요도)

In [ ]:
tr["oof"] = oof
ev_rmse = tr.groupby("event_id").apply(lambda d: rmse(d[TARGET], d["oof"])).sort_values(ascending=False)
sq = tr.assign(se=(tr[TARGET] - tr["oof"]) ** 2).groupby("event_id")["se"].sum().sort_values(ascending=False)
print("사상별 RMSE 분포"); print(ev_rmse.describe().round(3))
print(f"상위 5개 사상의 오차제곱합 비중: {sq.head(5).sum() / sq.sum():.1%}")

imp = pd.DataFrame({
    "feature": FEATURES,
    "gain": np.mean([m.feature_importance("gain") for m in models], axis=0),
}).sort_values("gain", ascending=False)
imp.head(30)

### 5-2. (선택) 데이터 소스별 Ablation
보고서용 "소스별 기여도" 비교. 4-fold를 그룹 수만큼 반복하므로 시간이 걸려 기본은 꺼둠.

In [ ]:
RUN_ABLATION = False

if RUN_ABLATION:
    static = ["cat_area", "wsarea", "mean_elevation", "mean_slope", "urban_ratio", "forest_ratio", "stream_order"]
    is_src = lambda f, key: key in f
    wl_feats = [f for f in FEATURES if ("wl" in f or f.startswith("dwl") or f.startswith("accel") or f.startswith("past_max"))
                and not f.startswith("xs_")]
    groups = {
        "수위만": wl_feats,
        "+IMERG": [f for f in FEATURES if is_src(f, "imerg") and not f.startswith("xs_")],
        "+SFC": [f for f in FEATURES if is_src(f, "sfc") and not f.startswith("xs_")],
        "+레이더": [f for f in FEATURES if is_src(f, "radar")],
        "+유역환경": static,
        "+동시각 공간": [f for f in FEATURES if f.startswith("xs_") or f.endswith("_vs_xs")],
    }
    base = ["station_id"] if USE_STATION_CAT else []
    used, rows = list(base), []
    for name, fs in groups.items():
        used += [f for f in fs if f not in used]
        s, _, _ = run_cv(used, verbose=False)
        rows.append({"단계": name, "피처 수": len(used), "OOF RMSE": round(s, 4)})
        print(rows[-1])
    display(pd.DataFrame(rows))

## STEP 6. Test 예측 · 제출 파일 생성

In [ ]:
te["pred"] = np.mean([m.predict(te[FEATURES], num_iteration=m.best_iteration) for m in models], axis=0)

submission = sub[["row_id"]].merge(te[["row_id", "pred"]], on="row_id", how="left")
submission = submission.rename(columns={"pred": "pred_maxrise_6h"})

assert len(submission) == len(sub), "행 수 불일치"
assert submission["row_id"].equals(sub["row_id"]), "row_id 순서 불일치"
assert submission["pred_maxrise_6h"].notna().all(), "결측 예측값 존재"

tag = "dev" if USE_COMMON_FOLDS else "all"
out_path = OUT_DIR / f"submission_lgb_{tag}_cv{cv_score:.4f}.csv"
submission.to_csv(out_path, index=False, encoding="utf-8")
print("저장:", out_path)
submission.describe()

## 다음 단계 후보
- 관측소 간 상·하류 지연 관계를 **train으로** 학습해 이웃 관측소 lag 피처 추가 (Q&A 10/6 허용)
- 미래 강우를 라벨로 한 보조모델 → 예측값을 피처로 (Q&A 10/6 허용, 방법론 문서에 명시)
- 감조 관측소 처리: 팀 기존 목록으로 플래그 추가
- 상위 오차 사상 집중 분석 (Global RMSE가 소수 대형 사상에 좌우됨)